# UR5e pi0-FAST LoRA Fine-Tuning

**All cells run in Google Colab Pro (A100 recommended) unless marked `[LAPTOP]`.**

## Requirements
- Google Colab Pro with A100 GPU (Runtime → Change runtime type → A100)
- HuggingFace account with a **write** token (`sheilsarda/pi0_ur5_fast_v1`, `sheilsarda/pi0_ur5_base_v1`)
- W&B account (wandb.ai)
- Dataset already on HuggingFace Hub: `sheilsarda/ur5_isaac_sim_v1`

## Current state
- `pi0_ur5_fast_v1`: 10k steps trained, checkpoint on HF Hub
- `pi0_ur5_base_v1`: not yet trained

In [ ]:
# [LAPTOP] — run this from your laptop, not Colab
#
# Prerequisites:
#   cd ~/Development/openpi && source .venv/bin/activate
#   huggingface-cli login   (paste your HF write token)
#
# Then run this cell, or paste the equivalent into a terminal:

import subprocess
result = subprocess.run([
    "python3", "-c",
    """
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
dataset = LeRobotDataset('sheilsarda/ur5_isaac_sim_v1')
dataset.push_to_hub(
    tags=['ur5e', 'isaac-sim'],
    private=False,
    push_videos=True,
    license='apache-2.0',
)
print('Done! Dataset live at https://huggingface.co/datasets/sheilsarda/ur5_isaac_sim_v1')
"""
], capture_output=False)


Done! Dataset live at https://huggingface.co/datasets/sheilsarda/ur5_isaac_sim_v1


In [1]:
# Cell 1: Verify GPU
!nvidia-smi

Sun Mar 29 05:07:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             45W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Cell 3: Authenticate HuggingFace (needed to download dataset AND upload checkpoints)
from huggingface_hub import login

# IMPORTANT: You must use a WRITE token to upload checkpoints later.
# Get your token from: https://huggingface.co/settings/tokens
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# Cell 4: Authenticate W&B
import wandb
wandb.login()  # paste your W&B API key

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sheilsarda to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
# Cell 4b: Authenticate Google Cloud
# Required to download pretrained weights from gs://openpi-assets/
from google.colab import auth
auth.authenticate_user()

In [22]:
# Cell 2: Clone repo and install dependencies
%cd /content/
!git clone https://github.com/sheilsarda/openpi.git

%cd openpi
!pip install uv -q
!uv sync

/content
Cloning into 'openpi'...
remote: Enumerating objects: 1295, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 1295 (delta 8), reused 11 (delta 7), pack-reused 1270 (from 2)
Receiving objects: 100% (1295/1295), 1.48 MiB | 4.22 MiB/s, done.
Resolving deltas: 100% (656/656), done.
/content/openpi
Using CPython 3.11.15
Creating virtual environment at: .venv
Resolved 281 packages in 0.83ms
Prepared 3 packages in 5.25s
Installed 242 packages in 175ms
 + absl-py==2.3.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.12.4
 + aiosignal==1.3.2
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + asttokens==3.0.0
 + attrs==25.3.0
 + augmax==0.4.1
 + av==14.4.0
 + beartype==0.19.0
 + beautifulsoup4==4.13.4
 + blinker==1.9.0
 + cachetools==5.5.2
 + certifi==2025.4.26
 + cffi==1.17.1
 + cfgv==3.4.0
 + charset-normalizer==3.4.2
 + chex==0.1.89
 + click==8.2.1
 + cloudpickle==3.1.1
 + cmake==4.0.2
 + comm==0.2.2
 + contourpy==

In [7]:
# so they survive the Colab session ending
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# Cell 5: Mount Google Drive — checkpoints will be saved here
import os
os.makedirs('/content/drive/MyDrive/openpi_checkpoints', exist_ok=True)


In [24]:

# Symlink so openpi writes checkpoints directly to Drive
os.symlink('/content/drive/MyDrive/openpi_checkpoints', '/content/openpi/checkpoints')

print('Checkpoints will be saved to: /content/drive/MyDrive/openpi_checkpoints')

Checkpoints will be saved to: /content/drive/MyDrive/openpi_checkpoints


---
## Training

Each model has three cells:
1. **Norm stats** — run once per model config (or after any dataset change); fast, CPU-only
2. **Train: FROM SCRATCH** — wipes existing checkpoints, starts from step 0
3. **Train: RESUME** — continues from the latest checkpoint; `--num-train-steps` is the new total

Run **one** of cells 2 or 3, not both.

---
### pi0-FAST (`pi0_ur5`)

In [25]:
# Norm stats — pi0-FAST
# Skip if already computed and the dataset hasn't changed.
!uv run scripts/compute_norm_stats.py --config-name=pi0_ur5

100% 4.07M/4.07M [00:02<00:00, 2.13MiB/s]
processor_config.json: 100% 253/253 [00:00<00:00, 3.10MB/s]
processing_action_tokenizer.py: 6.14kB [00:00, 46.7MB/s]
A new version of the following files was downloaded from https://huggingface.co/physical-intelligence/fast:
- processing_action_tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
tokenizer_config.json: 100% 322/322 [00:00<00:00, 6.28MB/s]
tokenizer.json: 687kB [00:00, 12.0MB/s]
special_tokens_map.json: 100% 3.00/3.00 [00:00<00:00, 36.8kB/s]
Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
episodes.jsonl: 100% 660/660 [00:00<00:00, 1.13MB/s]
Fetching 4 files:  25% 1/4 [00:00<00:00,  3.19it/s]
tasks.jsonl: 100% 45.0/45.0 [00:00<00:00, 681kB/s]

episodes_stats.jsonl: 25.8kB [00:00, 30.8MB/s]

info.json: 3.05kB [00:00, 29.4MB/s]
Fetching 4 files: 100% 4/4 [00:00<00:00, 11.65it/s]
Fetching 36 files:   0% 0/36 [00:00<?, ?it/s]
dat

In [26]:
# Train pi0-FAST — RESUME
# Continues from the latest checkpoint in checkpoints/pi0_ur5/ur5_fast_v1/.
# --num-train-steps is the TOTAL target (e.g. already at 10k → set 20k to run 10k more).
# W&B: resumes the existing ur5_fast_v1 run automatically.
!uv run scripts/train.py pi0_ur5 \
  --exp-name ur5_fast_v1 \
  --resume \
  --num-train-steps 20000

05:19:21.197 [I] Running on: 96f347217c1c                                                         (17575:train.py:213)
INFO:2026-03-29 05:19:21,397:jax._src.xla_bridge:925: Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
05:19:21.397 [I] Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig' (17575:xla_bridge.py:925)
INFO:2026-03-29 05:19:21,398:jax._src.xla_bridge:925: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
05:19:21.398 [I] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory (17575:xla_bridge.py:925)
05:19:21.946 [I] Created BasePyTreeCheckpointHandler: use_ocdbt=True, use_zarr3=False, pytree_metadata_options=PyTreeMetadataOptions(support_rich_types=False), array_metadata_store=<orbax.checkpo

In [27]:
# Upload pi0-FAST checkpoint to HuggingFace
# Uploads the full checkpoints/pi0_ur5/ur5_fast_v1/ folder (all saved steps).
from huggingface_hub import HfApi
import os

api = HfApi()
repo_id = 'sheilsarda/pi0_ur5_fast_v1'
api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)

# Infer the latest step from the checkpoint directory for the commit message.
ckpt_dir = '/content/drive/MyDrive/openpi_checkpoints/pi0_ur5/ur5_fast_v1'
steps = sorted(int(d) for d in os.listdir(ckpt_dir) if d.isdigit())
latest_step = steps[-1] if steps else '?'

api.upload_folder(
    folder_path=ckpt_dir,
    repo_id=repo_id,
    repo_type='model',
    commit_message=f'pi0-FAST LoRA ur5 checkpoint step {latest_step}',
)
print(f'Uploaded step {latest_step} → https://huggingface.co/{repo_id}')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5332dc0df1a8d7c80f082bbfc:   0%|          |  524kB /  636MB            

  ...f0c7ba4c10f0cbef0327618f7:  14%|#3        |  131MB /  946MB            

  ...e04fe64828194404dfee0cfe5:   1%|          | 20.7MB / 2.72GB            

...

  ...ef6f94e2b7a7e020fdbbc942e:   0%|          |  524kB / 2.15GB            

  ...8720a14b6487d8c551de0d6cd:   0%|          | 69.2kB / 99.6MB            

  ...ace294a5a90ace7c1b4478f67:   0%|          | 3.67MB /  861MB            

  ...e56e1959bc1132660e52e7fdb:  12%|#2        |  116MB /  946MB            

Uploaded step 19999 → https://huggingface.co/sheilsarda/pi0_ur5_fast_v1


# [LAPTOP] Download pi0-FAST checkpoint from HuggingFace for local serving


In [ ]:
#
# Prerequisites (run once):
#   cd ~/Development/openpi && source .venv/bin/activate
#   pip install huggingface_hub
#   huggingface-cli login   (read token is sufficient)
#
# Downloads only the step-15000 checkpoint subdirectory.

import logging
import sys
from huggingface_hub import snapshot_download

# Enable logging to see more details from the huggingface_hub library
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

print("Starting download...")

local_dir = '/home/sheil/Development/openpi/checkpoints/pi0_ur5/ur5_fast_v1'

path = snapshot_download(
    repo_id='sheilsarda/pi0_ur5_fast_v1',
    repo_type='model',
    local_dir=local_dir,
    allow_patterns='15000/**',
)

print(f"Download complete. Model saved to: {path}")
print()
print('To serve step 15000:')
print('  cd ~/Development/openpi')
print('  uv run scripts/serve_policy.py policy:checkpoint \\')
print('      --policy.config=pi0_ur5 \\')
print('      --policy.dir=checkpoints/pi0_ur5/ur5_fast_v1/15000')
